# CS781 Capstone Phase 1 — pretraining

Run this on Kaggle with a GPU.

1. Create a notebook, set the accelerator to GPU T4, and upload this file.
2. Add `legal_bpe.py` and `legal_lm.py` with File → Add utility script. Both must sit in `/kaggle/working`.
3. Add a secret named `HF` with your Hugging Face token. Accept the dataset terms for `Exploration-Lab/CS-781-Capstone` on the Hugging Face website first.
4. Save Version → Save & Run All (GPU).
5. From the Output tab, download `submission.zip` and submit that zip. It contains `prediction.json`.

This notebook trains a decoder I wrote in `legal_lm.py` on Supreme Court text, using the byte-level BPE in `legal_bpe.py`. It does not load GPT-2 weights. `TRAIN_TOKENS` is set so one GPU session can finish and write the zip. If the run ends with time left, raise `TRAIN_TOKENS` and run again.


In [ ]:
import os
import sys
import math
import json
import time
import random
import zipfile
from pathlib import Path

import torch
from huggingface_hub import login, hf_hub_download
from kaggle_secrets import UserSecretsClient

for folder in (Path.cwd(), Path("/kaggle/working")):
    if (folder / "legal_bpe.py").exists():
        sys.path.insert(0, str(folder))
        break

from legal_bpe import LegalBPE, iter_texts
from legal_lm import LegalLM, count_parameters

HF_TOKEN = UserSecretsClient().get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
print("logged in")
print("cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")


In [ ]:
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_FILE = "Phase_1_Pretraining/Train/train_supreme_court.json"
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

BPE_VOCAB_SIZE = 8192
BPE_MIN_PAIR_COUNT = 2
BPE_SCORE = "dice"
BPE_MAX_CHARS = 30_000_000

TRAIN_TOKENS = 50_000_000
VAL_TOKENS = 400_000
BLOCK = 256
N_EMBD = 384
N_LAYER = 6
N_HEAD = 6
BATCH = 16
GRAD_ACCUM = 4
EPOCHS = 1
LR = 3e-4
WARMUP = 200
WEIGHT_DECAY = 0.1
EVAL_EVERY = 200
MAX_NEW = 320
TEMPERATURE = 0.8
TOP_P = 0.9
TARGET_WORDS = 128
GEN_BATCH = 8

OUTPUT_DIR = "/kaggle/working/cs781_phase1"
os.makedirs(OUTPUT_DIR, exist_ok=True)
BPE_PATH = os.path.join(OUTPUT_DIR, "legal_bpe.json")
CKPT_PATH = os.path.join(OUTPUT_DIR, "legal_lm.pt")
PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")
LOSS_PLOT_PATH = os.path.join(OUTPUT_DIR, "training_validation_loss.png")
print("config ready", OUTPUT_DIR)


In [ ]:
train_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TRAIN_FILE,
    repo_type="dataset",
    token=HF_TOKEN,
)
print("train file", train_path)

with open(train_path, "r", encoding="utf-8") as handle:
    records = json.load(handle)

if os.path.exists(BPE_PATH):
    tokenizer = LegalBPE.load(BPE_PATH)
    print("loaded tokenizer", len(tokenizer))
else:
    tokenizer = LegalBPE.train(
        iter_texts(records),
        vocab_size=BPE_VOCAB_SIZE,
        min_pair_count=BPE_MIN_PAIR_COUNT,
        score=BPE_SCORE,
        max_chars=BPE_MAX_CHARS,
    )
    tokenizer.save(BPE_PATH)

for sample in (
    "The Supreme Court held that the petitioner",
    "Section 302 of the Indian Penal Code",
):
    ids = tokenizer.encode(sample)
    print(len(ids), tokenizer.decode(ids))


In [ ]:
train_ids = []
val_ids = []
random.seed(SEED)
separator = [tokenizer.eos_token_id]

for text in iter_texts(records):
    piece = tokenizer.encode(text, add_special_tokens=False) + separator
    if random.random() < 0.05 and len(val_ids) < VAL_TOKENS:
        val_ids.extend(piece[: VAL_TOKENS - len(val_ids)])
    elif len(train_ids) < TRAIN_TOKENS:
        train_ids.extend(piece[: TRAIN_TOKENS - len(train_ids)])
    if len(train_ids) >= TRAIN_TOKENS and len(val_ids) >= VAL_TOKENS:
        break

del records

def to_rows(flat):
    usable = (len(flat) // BLOCK) * BLOCK
    rows = usable // BLOCK
    data = torch.tensor(flat[:usable], dtype=torch.long).view(rows, BLOCK)
    return data

train_data = to_rows(train_ids)
val_data = to_rows(val_ids)
del train_ids, val_ids
print("train rows", len(train_data), "val rows", len(val_data))


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LegalLM(
    vocab_size=len(tokenizer),
    n_embd=N_EMBD,
    n_layer=N_LAYER,
    n_head=N_HEAD,
    block_size=BLOCK,
).to(device)
print(f"parameters {count_parameters(model) / 1e6:.2f}M")

use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler(device.type, enabled=use_amp)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95))

rows_per_step = BATCH * GRAD_ACCUM
steps = EPOCHS * max(1, len(train_data) // rows_per_step)

def lr_at(step):
    if step < WARMUP:
        return LR * (step + 1) / max(1, WARMUP)
    progress = (step - WARMUP) / max(1, steps - WARMUP)
    return LR * 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))

def batch_loss(data, index):
    take = min(BATCH, len(data) - index)
    batch = data[index : index + take].to(device, non_blocking=True)
    with torch.amp.autocast(device_type=device.type, enabled=use_amp):
        _logits, loss = model(batch, labels=batch)
    return loss

@torch.no_grad()
def evaluate():
    model.eval()
    total = 0.0
    count = 0
    limit = min(len(val_data), BATCH * 20)
    for start in range(0, limit, BATCH):
        loss = batch_loss(val_data, start)
        total += float(loss) * (min(BATCH, limit - start))
        count += min(BATCH, limit - start)
    model.train()
    mean = total / max(1, count)
    return mean, math.exp(min(20.0, mean))

order = torch.randperm(len(train_data))
cursor = 0
train_hist = []
val_hist = []
model.train()
started = time.time()

for step in range(steps):
    lr = lr_at(step)
    for group in opt.param_groups:
        group["lr"] = lr
    opt.zero_grad(set_to_none=True)
    running = 0.0
    for _micro in range(GRAD_ACCUM):
        if cursor + BATCH > len(order):
            order = torch.randperm(len(train_data))
            cursor = 0
        pick = order[cursor : cursor + BATCH]
        cursor += BATCH
        batch = train_data[pick].to(device, non_blocking=True)
        with torch.amp.autocast(device_type=device.type, enabled=use_amp):
            _logits, loss = model(batch, labels=batch)
            loss = loss / GRAD_ACCUM
        scaler.scale(loss).backward()
        running += float(loss.detach()) * GRAD_ACCUM
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(opt)
    scaler.update()
    train_hist.append(running / GRAD_ACCUM)
    if step % 50 == 0 or step + 1 == steps:
        print(f"step {step + 1}/{steps} loss {train_hist[-1]:.4f} lr {lr:.6f}")
    if step % EVAL_EVERY == 0 or step + 1 == steps:
        val_loss, ppl = evaluate()
        val_hist.append((step, val_loss))
        print(f"val loss {val_loss:.4f} ppl {ppl:.2f}")
        torch.save(
            {"model": model.state_dict(), "step": step, "vocab": len(tokenizer)},
            CKPT_PATH,
        )

print(f"trained in {(time.time() - started) / 3600:.2f} hours")


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(train_hist, label="train loss")
if val_hist:
    plt.plot([item[0] for item in val_hist], [item[1] for item in val_hist], label="val loss")
plt.xlabel("optimizer step")
plt.ylabel("loss")
plt.title("Phase 1 loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(LOSS_PLOT_PATH, dpi=150)
plt.show()
print("plot", LOSS_PLOT_PATH)


In [ ]:
test_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TEST_FILE,
    repo_type="dataset",
    token=HF_TOKEN,
)
with open(test_path, "r", encoding="utf-8") as handle:
    test_data = json.load(handle)

model.eval()
items = list(test_data.items())
predictions = {}
max_prompt = BLOCK - 1

for start in range(0, len(items), GEN_BATCH):
    batch = items[start : start + GEN_BATCH]
    prompts = []
    example_ids = []
    for example_id, record in batch:
        example_ids.append(str(example_id))
        if isinstance(record, dict):
            text = record.get("text") or ""
        else:
            text = str(record)
        prompts.append(text if text.strip() else ".")
    encoded, _mask = tokenizer.batch_encode(prompts, max_length=max_prompt, padding="right")
    idx = torch.tensor(encoded, dtype=torch.long, device=device)
    with torch.inference_mode():
        out, prompt_lengths, lengths, produced = model.generate(
            idx,
            max_new_tokens=MAX_NEW,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            eos_id=tokenizer.eos_token_id,
            pad_id=tokenizer.pad_token_id,
        )
    for row, example_id in enumerate(example_ids):
        new_tokens = produced[row]
        text = tokenizer.decode(new_tokens, skip_special_tokens=True)
        words = text.split()[:TARGET_WORDS]
        predictions[example_id] = {"text": " ".join(words)}
    done = min(start + GEN_BATCH, len(items))
    print(f"generated {done}/{len(items)}")

with open(PREDICTION_PATH, "w", encoding="utf-8") as handle:
    json.dump(predictions, handle, ensure_ascii=False, indent=2)

with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    bundle.write(PREDICTION_PATH, arcname="prediction.json")

print("zip", SUBMISSION_ZIP, "examples", len(predictions))
